In [ ]:
import os
import operator
import OpenDartReader
import yfinance as yf
import pandas as pd
from datetime import datetime
from typing import Annotated, List
from typing_extensions import TypedDict
from pydantic import BaseModel, Field
from langchain_openai import ChatOpenAI
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph import StateGraph, END
from langgraph.prebuilt import create_react_agent
from langgraph.checkpoint.memory import MemorySaver
from langchain_core.tools import tool
from langchain_teddynote import logging
from langchain_core.runnables import RunnableConfig
from langchain_teddynote.messages import random_uuid, invoke_graph
from docling.document_converter import DocumentConverter
from tavily import TavilyClient
from dotenv import load_dotenv
import json

# API KEY 정보로드
load_dotenv()

# 프로젝트 이름을 입력합니다.
logging.langsmith(os.environ["STUDENT_NAME"] + "-" + "Supervisor_Agent")

print("##################### Completed Cell #####################")

In [ ]:
# 사용자 투자 프로프일 로딩
def get_user_profile():

    with open("./data/Your_profile.json", "r", encoding="utf-8") as f:
        profile_data = json.load(f)

    profile_str = json.dumps(profile_data, ensure_ascii=False, indent=2)

    return profile_str

print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 1. 모델 설정 (API Key 유출 방지를 위해 환경 변수 사용 권장)
# ---------------------------------------------------------
llm = ChatOpenAI(
    model="gpt-4o",
    temperature=0.1,  # 라우팅의 정확도와 일관성을 위해 0으로 고정합니다.
    api_key=os.environ["OPENAI_API_KEY"],
)


# ---------------------------------------------------------
# 2. 상태(State) 값 클래스 정의
# ---------------------------------------------------------
class AgentState(TypedDict):
    messages: Annotated[List[BaseMessage], operator.add]
    next: str


# ---------------------------------------------------------
# 3. 4개 주식투자 전문 Worker 에이전트 노드 생성
# ---------------------------------------------------------
def create_worker_agent(system_message: str, tools: list = []):
    return create_react_agent(
        model=llm, tools=tools, state_modifier=SystemMessage(content=system_message)
    )

print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 4. 경제데이터 조회 툴
# ---------------------------------------------------------
@tool
def Economy_Data_Tool() -> str:
    """
    투자 환경 판단을 위해 5개의 주요 경기 지표등을 조회 합니다.
    조회 항목: VIX, CLI, 환율, BDI, WTI, 장단기금리차
    """
    file_path = "./data/경제지표_regime_1.pdf"

    # 1. 파일 존재 여부 확인
    if not os.path.exists(file_path):
        return f"오류: 지정된 경로에 '{file_path}' 파일이 존재하지 않습니다. 현재 경로를 확인해 주세요."

    try:
        # 2. Docling 변환기 객체 생성 및 파일 파싱
        converter = DocumentConverter()
        result = converter.convert(file_path)

        # 3. 문서 구조를 마크다운 텍스트로 변환 (표 구조가 가장 잘 보존됨)
        markdown_context = result.document.export_to_markdown()

        # 4. 파싱된 내용이 비어있을 경우 예외 처리
        if not markdown_context.strip():
            return "파일 파싱은 성공했으나, 추출된 텍스트 컨텍스트가 비어 있습니다."

        return markdown_context

    except Exception as e:
        # 5. 실행 중 에러 발생 시 디버깅을 위한 메시지 리턴
        return f"Docling으로 PDF를 읽는 중 오류가 발생했습니다: {str(e)}"
    

print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 4. 기업 재무정보 조회 툴
# ---------------------------------------------------------


@tool
def Company_Fin_Tool(company: str, year: int = None) -> str:
    """
    OpenDartReader를 사용하여 특정 기업의 주요 재무 지표를 종합 조회합니다.
    조회 항목: 매출액, 영업이익, 당기순이익, 자산총계, 부채총계

    Args:
        company (str): 조회하고자 하는 기업명 (예: '삼성전자') 또는 종목코드 (예: '005930')
        year (int, optional): 조회하고자 하는 결산 연도 4자리. 입력하지 않으면 직전 연도가 디폴트로 설정됩니다.

    Returns:
        str: 해당 기업 및 연도의 주요 재무 상태 및 손익 요약 정보 데이터
    """
    # 1. API 키 설정 (환경변수 또는 지정된 키)
    api_key = os.environ["DART_API_KEY"]

    # 2. 연도 미입력 시 직전 연도를 디폴트로 설정?
    year = datetime.now().year - 1

    try:
        # 3. OpenDartReader 객체 생성 및 데이터 조회 (11011: 연간 사업보고서)
        dart = OpenDartReader(api_key)
        df = dart.finstate(company, year, reprt_code="11011")

        if df is None or df.empty:
            return f"안내: {company}의 {year}년도 연간 사업보고서 데이터가 아직 등록되지 않았거나 기업명을 찾을 수 없습니다."

        # 재무 항목 초기화
        financial_data = {
            "매출액": "데이터 없음",
            "영업이익": "데이터 없음",
            "당기순이익": "데이터 없음",
            "자산총계": "데이터 없음",
            "부채총계": "데이터 없음",
        }
        actual_corp_name = company

        # 4. 데이터프레임 순회하며 주요 항목 매칭
        for _, row in df.iterrows():
            actual_corp_name = row.get("corp_name", company)
            sj_div = row.get("sj_div", "")  # BS: 재무상태표, IS/CIS: 손익계산서
            account_nm = str(row.get("account_nm", "")).replace(" ", "")
            amount_str = row.get("thstrm_amount", "0")

            # 금액 천단위 콤마 및 통화 포맷팅
            try:
                formatted_amount = f"{int(str(amount_str).replace(',', '')):,} 원"
            except ValueError:
                formatted_amount = f"{amount_str} 원"

            # [재무상태표 항목] 자산 및 부채
            if sj_div == "BS":
                if "자산총계" in account_nm:
                    financial_data["자산총계"] = formatted_amount
                elif "부채총계" in account_nm:
                    financial_data["부채총계"] = formatted_amount

            # [손익계산서 항목] 매출, 영업이익, 당기순이익
            elif sj_div in ["IS", "CIS"]:
                if "매출액" in account_nm or account_nm == "매출":
                    financial_data["매출액"] = formatted_amount
                elif "영업이익" in account_nm:
                    financial_data["영업이익"] = formatted_amount
                elif "당기순이익" in account_nm:
                    financial_data["당기순이익"] = formatted_amount

        # 5. LLM 에이전트 가독성을 위한 마크다운 포맷팅
        result_summary = (
            f"### [{actual_corp_name}] {year}년도 정기 사업보고서 주요 재무지표\n"
            f"- **조회 대상 기업**: {company}\n\n"
            f"#### 손익계산서 (Income Statement)\n"
            f"- **연간 매출액**: {financial_data['매출액']}\n"
            f"- **연간 영업이익**: {financial_data['영업이익']}\n"
            f"- **연간 당기순이익**: {financial_data['당기순이익']}\n\n"
            f"#### 재무상태표 (Balance Sheet)\n"
            f"- **자산 총계**: {financial_data['자산총계']}\n"
            f"- **부채 총계**: {financial_data['부채총계']}\n"
        )
        return result_summary

    except Exception as e:
        return f"OpenDartReader 실행 중 오류가 발생했습니다: {str(e)}"
    

print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 4. 기업 주식정보 조회 툴
# ---------------------------------------------------------


@tool
def Stock_Data_Tool(ticker: str) -> str:
    """
    야후 파이낸스(Yahoo Finance)에서 특정 주식 종목의 실시간 시세,
    핵심 투자 지표, 최신 재무 스냅샷 및 시장 분석가 의견을 종합적으로 조회합니다.

    Args:
        ticker (str): 분석할 주식의 티커 심볼 (예: 미국주식 'AAPL', 'TSLA', 한국주식 '005930.KS')

    Returns:
        str: 해당 종목의 투자 분석용 통합 마크다운 컨텍스트 데이터
    """
    try:
        # 1. Ticker 객체 생성
        stock = yf.Ticker(ticker.strip().upper())
        info = stock.info

        if not info or "symbol" not in info:
            return f"오류: 티커 '{ticker}'에 해당하는 주식 종목 정보를 야후 파이낸스에서 찾을 수 없습니다."

        # 2. 기본 기업 개요 데이터 추출
        comp_name = info.get("longName", info.get("shortName", ticker))
        currency = info.get("currency", "USD")
        current_price = info.get(
            "currentPrice", info.get("regularMarketPrice", "데이터 없음")
        )

        # 3. 핵심 가치 평가 및 투자 지표 (Fundamental Metrics)
        market_cap = info.get("marketCap", 0)
        trailing_per = info.get("trailingPE", "데이터 없음")
        forward_per = info.get("forwardPE", "데이터 없음")
        price_to_book = info.get("priceToBook", "데이터 없음")
        dividend_yield = (
            info.get("dividendYield", 0) * 100 if info.get("dividendYield") else 0
        )

        # 시가총액 가독성 처리 (조/억 단위 변환 등 기본 포맷팅)
        formatted_market_cap = f"{market_cap:,}" if market_cap else "데이터 없음"

        # 6. LLM이 심층 추론(Reasoning)하기 좋은 형태의 통합 보고서 문서 포맷팅
        analysis_report = (
            f"### [{comp_name} ({info.get('symbol')})] 야후 파이낸스 투자 지표 분석\n\n"
            f"#### 실시간 시세 정보\n"
            f"- **현재가**: {current_price} {currency}\n"
            f"- **시가총액**: {formatted_market_cap} {currency}\n\n"
            f"#### 핵심 투자 밸류에이션 지표\n"
            f"- **트레일링 PER (지난 12개월)**: {trailing_per}\n"
            f"- **포워드 PER (향후 12개월 예상)**: {forward_per}\n"
            f"- **PBR (주가순자산비율)**: {price_to_book}\n"
            f"- **배당수익률 (Dividend Yield)**: {dividend_yield:.2f}%\n\n"
        )
        return analysis_report

    except Exception as e:
        return (
            f"야후 파이낸스에서 데이터를 가져오는 중 예외 에러가 발생했습니다: {str(e)}"
        )
        

print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 4. 기업 뉴스정보 조회 툴
# ---------------------------------------------------------


@tool
def News_Data_Tool(query: str) -> str:
    """
    Tavily Search API를 사용하여 사용자의 질문과 관련된 최신 경제, 투자, 기업 뉴스를 검색합니다.
    실시간 시장 동향이나 기업 이슈를 파악할 때 호출하세요.

    Args:
        query (str): 검색할 경제/투자/기업 관련 핵심 키워드 또는 문장 (예: '엔비디아 주가 전망', '국내 금리 인하 영향')

    Returns:
        str: 최신 뉴스 제목, 요약, 출처 URL이 포함된 마크다운 구조의 컨텍스트 데이터
    """

    try:
        # 2. Tavily 클라이언트 객체 생성
        tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

        # 3. 뉴스 특화 검색 실행
        # topic="news"를 지정하면 블로그나 일반 웹페이지를 제외한 주요 언론사 기사 위주로 검색합니다.
        response = tavily_client.search(
            query=query.strip(),
            topic="news",
            max_results=5,
            include_answer=False,  # AI 요약문 대신 기사 원문 컨텍스트 위주로 수집
        )

        results = response.get("results", [])
        if not results:
            return f"안내: Tavily 뉴스 검색에서 '{query}'와 관련된 최신 결과를 찾을 수 없습니다."

        # 4. LLM 에이전트 가독성을 위한 마크다운 포맷팅
        current_date = datetime.now().strftime("%Y-%m-%d")
        result_summary = (
            f"### '{query}' 관련 실시간 뉴스 검색 결과 (검색일: {current_date})\n\n"
        )

        for i, res in enumerate(results):
            title = res.get("title", "뉴스 기사")
            url = res.get("url", "#")
            content = res.get("content", "내용 요약 정보가 없습니다.")
            # 뉴스 검색 시 제공되는 발행일 정보 파싱 (제공되지 않을 경우 기본값 처리)
            pub_date = res.get("published_date", "최근 발행")

            result_summary += f"{i+1}. **[{title}]({url})**\n"
            result_summary += f"   - 발행일: {pub_date}\n"
            result_summary += f"   - 요약: {content}\n\n"

        return result_summary

    except Exception as e:
        return f"Tavily 라이브러리 실행 중 오류가 발생했습니다: {str(e)}"
    

print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 5. 5개의 Agent를 생성 한다.
# ---------------------------------------------------------

worker_1_tools = [Economy_Data_Tool]
worker_2_tools = [Company_Fin_Tool]
worker_3_tools = [Stock_Data_Tool]
worker_4_tools = [News_Data_Tool]

# Worker 1: 경제분석 전문가
worker_1 = create_worker_agent(
    "당신은 거시경제(매크로) 시장 분석 전문가입니다. VIX, CLI, 환율, BDI, WTI, 장단기금리차 등 글로벌 경기 지표 등을 분석하여 현재 투자 환경을 평가하세요.",
    worker_1_tools,
)
# Worker 2: 기업분석 전문가
worker_2 = create_worker_agent(
    "당신은 기업정보 분석 전문가입니다. 해당 기업의 비즈니스 모델, 재무제표(매출, 영업이익), 시장 점유율, 경쟁사 대비 강점을 분석하세요.",
    worker_2_tools,
)
# Worker 3: 주가분석 전문가
worker_3 = create_worker_agent(
    "당신은 기술적 및 계량적 주가 분석 전문가입니다. 주가 추세, 차트 패턴, 거래량, 주요 이동평균선 및 밸류에이션(PER, PBR) 지표를 분석하세요.",
    worker_3_tools,
)

# Worker 4: 뉴스분석 전문가
worker_4 = create_worker_agent(
    "당신은 경제 뉴스 분석 전문가입니다. 관심 섹터/기업/종목 등의 최신 경제뉴스를 분석해 투자관점 이슈와 이벤트를 분석하세요.",
    worker_4_tools,
)
# Worker 5: 투자전략 전문가
worker_5 = create_worker_agent(
    """
    당신은 투자전략 보고서 작성 전문가입니다. 
    앞선 매크로 시장 분석, 기업정보 분석, 주가 분석, 뉴스 분석 등을 종합하여 
    국내외 마켓 상황 및 리스크 요인, 명확한 투자 의견(매수/매도/홀딩)이 포함된 
    '최종 투자전략 보고서'를 아래 '사용자 프로파일' 정보도 고려해 완성도 높게 작성하세요.
    
    [사용자 프로파일]
    
    """ + get_user_profile(),
)


print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 6. 노드 정의 및 실행 함수
# ---------------------------------------------------------


def run_worker_node(agent, name: str):
    def node(state: AgentState):
        result = agent.invoke(state)
        final_message = result["messages"][-1]
        return {
            "messages": [
                AIMessage(
                    content=f"[{name}의 분석 결과]:\n{final_message.content}", name=name
                )
            ]
        }

    return node


# ---------------------------------------------------------
# 7. Fallback 노드 정의 (주식투자 자문 목적 외의 질문 처리)
# ---------------------------------------------------------
def fallback_node(state: AgentState):
    fallback_message = AIMessage(
        content="죄송합니다. 저는 주식투자 자문 및 자산 분석을 수행하는 'AI 투자 메이트' 에이전트입니다. 경제, 주식, 기업, 뉴스 분석과 관련 없는 질문에는 답변을 제공할 수 없습니다. 좀더 투자 관점의 명확한 질문을 해주세요. "
    )
    return {"messages": [fallback_message], "next": "FINISH"}


print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 8. Supervisor 에이전트 노드 생성 (AI 투자 메이트 팀 매니저)
# ---------------------------------------------------------
class SupervisorRouter(BaseModel):
    next: str = Field(
        description=(
            "다음에 작업을 위임할 Worker 이름 ('Economy_Analysis_Expert', 'Company_Analysis_Expert', 'Stock_Analysis_Expert', 'News_Analysis_Expert', 'Invest_Analysis_Expert'). "
            "종합 보고서 작성이 완료되어 모든 프로세스가 끝났다면 'FINISH'. "
            "질문이 주식 투자, 경제, 기업, 뉴스 분석 등과 전혀 관련 없는 일상 대화, 잡담, 프로그래밍, 일상 질문이라면 반드시 'Fallback'을 선택하세요."
        ),
    )


def supervisor_node(state: AgentState):
    messages = state["messages"]

    supervisor_prompt = """
    당신은 주식투자 자문 시스템의 총괄 리더인 'AI 투자 메이트 매니저'입니다.
    사용자의 투자 관련 요청을 파악하고 아래 5명의 분석 전문가들을 지휘합니다.
    분석가들의 결과가 모순이 있는지 논리적으로 검토하면서, 최상의 투자 전략을 도출하세요:  
    
    1. Economy_Analysis_Expert: 매크로(거시경제) 분석 (VIX, CLI, 환율, BDI, WTI, 장단기금리차 등 거시경제 상황 파악)
    2. Company_Analysis_Expert: 기업정보 분석 (재무제표, 사업 모델, 기업 펀더멘탈 파악)
    3. Stock_Analysis_Expert: 주식 분석 (차트, 추세, 밸류에이션, 가격 지표 파악)
    4. News_Analysis_Expert: 뉴스 분석 (경제 뉴스로 부터 투자 이벤트 파악)
    5. Invest_Analysis_Expert: 투자전략 보고서 생성 (Economy_Analysis_Expert, Company_Analysis_Expert, Stock_Analysis_Expert, News_Analysis_Expert 의 모든 분석 결과를 종합하여 최종 리포트 작성)
    
    [업무 프로세스 및 라우팅 규칙]
    - 주식 투자와 완전히 무관한 질문이나 사적인 대화가 들어오면 즉시 'Fallback'을 선택하세요.
    - 전문가들의 분석 결과가 모순이 있고, 논리적이지 않다면 이전 Worker를 다시 수행 하세요.
    - 분석 요청이 들어오면 4명(Economy_Analysis_Expert, Company_Analysis_Expert, Stock_Analysis_Expert, News_Analysis_Expert)의 분석 전문가들에게 의뢰하고 의견을 취합 후, 최종적으로 Invest_Analysis_Expert(보고서 생성)에게 일을 넘겨 종합 결과물을 만들어야 합니다.
    - Invest_Analysis_Expert가 최종 투자전략 보고서 작성을 마쳐 완벽한 자문 답변이 준비되었다면 'FINISH'를 선택하세요.
  
    """

    # TypeError 방지를 위한 .with_structured_output() 구조화 출력 보장
    structured_llm = llm.with_structured_output(SupervisorRouter)

    try:
        response = structured_llm.invoke(
            [SystemMessage(content=supervisor_prompt)] + messages
        )
        next_agent = response.next
    except Exception as e:
        print(f"[Supervisor 오류 발생]: {e} -> 'Fallback'으로 안전하게 우회합니다.")
        next_agent = "Fallback"

    return {"next": next_agent}


print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 9. 그래프 정의 및 노드/엣지 연결 (StateGraph)
# ---------------------------------------------------------
workflow = StateGraph(AgentState)

# 노드 등록 (4개 Worker 및 Fallback 노드 확장)
workflow.add_node("Supervisor", supervisor_node)
workflow.add_node(
    "Economy_Analysis_Expert", run_worker_node(worker_1, "Economy_Analysis_Expert")
)
workflow.add_node(
    "Company_Analysis_Expert", run_worker_node(worker_2, "Company_Analysis_Expert")
)
workflow.add_node(
    "Stock_Analysis_Expert", run_worker_node(worker_3, "Stock_Analysis_Expert")
)
workflow.add_node(
    "News_Analysis_Expert", run_worker_node(worker_4, "News_Analysis_Expert")
)
workflow.add_node(
    "Invest_Analysis_Expert", run_worker_node(worker_5, "Invest_Analysis_Expert")
)
workflow.add_node("Fallback", fallback_node)

# 진입점 설정
workflow.set_entry_point("Supervisor")

# 조건부 엣지 매핑 정보 설정
workflow.add_conditional_edges(
    "Supervisor",
    lambda x: x["next"],
    {
        "Economy_Analysis_Expert": "Economy_Analysis_Expert",
        "Company_Analysis_Expert": "Company_Analysis_Expert",
        "Stock_Analysis_Expert": "Stock_Analysis_Expert",
        "News_Analysis_Expert": "News_Analysis_Expert",
        "Invest_Analysis_Expert": "Invest_Analysis_Expert",
        "Fallback": "Fallback",
        "FINISH": END,
    },
)

# 모든 Worker들은 업무 후 매니저에게 돌아와 컨펌을 받음
workflow.add_edge("Economy_Analysis_Expert", "Supervisor")
workflow.add_edge("Company_Analysis_Expert", "Supervisor")
workflow.add_edge("Stock_Analysis_Expert", "Supervisor")
workflow.add_edge("News_Analysis_Expert", "Supervisor")
workflow.add_edge("Invest_Analysis_Expert", "Supervisor")
workflow.add_edge("Fallback", END)


print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 10. 그래프 컴파일 및 시각화
# ---------------------------------------------------------
from langchain_teddynote.graphs import visualize_graph

memory = MemorySaver()
graph = workflow.compile(checkpointer=memory)
visualize_graph(graph)


print("##################### Completed Cell #####################")

In [ ]:
# ---------------------------------------------------------
# 12. 사용자 메시지 입력
# ---------------------------------------------------------

config = RunnableConfig(recursion_limit=15, configurable={"thread_id": random_uuid()})

inputs = {
    "messages": [HumanMessage(content="삼성전자 주식 매도 타이밍 분석해줘.")],
}

invoke_graph(graph, inputs, config)